In [ ]:
import  pandas as pd
df=pd.read_csv('данные\\free')

In [23]:
import pandas as pd
df=pd.read_csv('result.csv')
df

,дата,время,рейтинг,текст,автор,likes_count,official_answer,отделение
0,2026-04-15,23:57:46,1,Долго обслуживают,Тимур Искаков,0,NaN,"Freedom Bank, Курмангазы"
1,2026-04-15,22:18:37,1,"Добрый день, приобрела билет в вашем приложени...",Наталья Ким,0,{'date_created': '2026-04-16T04:31:57.607004+0...,"Freedom Bank, Курмангазы"
2,2026-04-15,18:07:18,1,Очень медленно работают \nГрубят \nНет места д...,Мурат Хайдаров,0,{'date_created': '2026-04-16T04:29:03.021833+0...,"Freedom Bank, Курмангазы"
3,2026-04-13,21:00:52,1,Просидел в очереди два часа при минимальном ко...,Grishkofan,0,NaN,"Freedom Bank, Курмангазы"
4,2026-04-11,14:42:17,1,Добрый день! Сразу по сервису: \n- в комнатке ...,Alexandr Turdiyev,0,{'date_created': '2026-04-13T07:46:03.203688+0...,"Freedom Bank, Курмангазы"
...,...,...,...,...,...,...,...,...
5193,2024-08-15,20:06:28,1,"ужасно обслуживание, менеджера элементарную по...",Берик Елеусизов,2,{'date_created': '2024-08-18T10:41:51.308397+0...,"Freedon Bank, Достык"
5194,2024-08-06,12:16:33,2,Слабые менеджеры,Adlet Shakenov,2,{'date_created': '2024-08-06T21:35:40.561494+0...,"Freedon Bank, Достык"
5195,2024-08-01,14:17:29,5,Хочу выразить благодарность кассиру Мукажанова...,Albina Marat,2,{'date_created': '2025-04-28T20:31:24.995252+0...,"Freedon Bank, Достык"
5196,2024-07-24,17:00:41,5,Лучши Банк!,Apple User,1,{'date_created': '2024-07-24T20:55:03.041903+0...,"Freedon Bank, Достык"


In [26]:
import spacy
import pandas as pd

nlp = spacy.load("ru_core_news_sm")  # если отзывы на русском

def extract_persons(text):
    if not isinstance(text, str):
        return []
    
    doc = nlp(text)
    
    return [
        ent.text.strip()
        for ent in doc.ents
        if ent.label_ == "PER" or ent.label_ == "PERSON"
    ]

df["люди"] = df["текст"].apply(extract_persons)

In [171]:
df.to_csv("cleaned_reviews.csv", index=False, encoding="utf-8-sig")

In [102]:
df=pd.read_csv("cleaned_reviews.csv")

In [103]:
import ast
import re

df["people"] = df["люди"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
def clean_name(name):
    name = name.strip()

    # убрать казахские/русские падежные окончания грубо
    name = re.sub(r"(ға|ге|да|де|ты|ті|у|ю)$", "", name)

    return name
df["people"] = df["people"].apply(
    lambda lst: [clean_name(x) for x in lst]
)

def normalize_name(name):
    
    mapping = {
        "Жунусбаева Әсем": "Жунусбаева Асем",
        "Жунусбева Әсем Бақытханқызы": "Жунусбаева Асем",
        "Жайлаубаев Дастан": "Жайлаубаев Дастан",
        "Жайлаубаев Дастанға": "Жайлаубаев Дастан",
        "Жайлаубаеву Дастану": "Жайлаубаев Дастан",
        "Алтаева": "Алтаева Жазира",
        "Алтаева Жазира": "Алтаева Жазира",
        'Жазира Арыстанбекова': 'Алтаева Жазира',
        'Арыстанбекова Жазира': 'Алтаева Жазира',
        'Махмуд': 'Махмуд Альбина',
        'Махмуд Альбине': 'Махмуд Альбина'
    }
    return mapping.get(name, name)

df["people"] = df["people"].apply(
    lambda lst: [normalize_name(x) for x in lst]
)

In [106]:
from rapidfuzz import process

canonical = ["Жунусбаева Асем", "Жайлаубаев Дастан", "Алтаева Жазира", 'Айгерим', 'Махмуд Альбина', 'Калибекова Айнура']

from rapidfuzz import process, fuzz

names = df["people"].explode().dropna().unique()
df = df.explode("people")

mapping = {}
canonical = []

for name in names:
    if not canonical:
        canonical.append(name)
        mapping[name] = name
        continue

    match, score, _ = process.extractOne(name, canonical, scorer=fuzz.WRatio)

    if score > 90:
        mapping[name] = match
    else:
        canonical.append(name)
        mapping[name] = name

df["people"] = df["people"].map(mapping)

In [107]:
print(df['people'].value_counts().head(25))


people
Жунусбаева Асем              662
Алтаева Жазира               118
Дастан Жайлаубаев\nХорошо    113
Фридом                        93
Жайлаубаеву Дастан            52
Калибекова Айнура             44
Айгерим                       44
Өте                           41
Ерсултан                      35
Дуйсенова Айгерим             33
Байбосын Акерке               31
Айзат                         31
Жазира Арыстанбековна         29
Кудайбергенова Гаухар         27
Ергазмна Таншолпан            26
Махмуд Альбина                25
Ильяс                         25
Омаровой Айгерим              24
Еркебулан                     22
Кожахметову Айзат             22
Рахмет                        21
Турсагуловой Асем             21
Айганым                       21
Турсагулова Әсем              21
Калибекова                    21
Name: count, dtype: int64


In [110]:
def normalize_kz(name):
    if isinstance(name, str):
        name = name.lower()
        replacements = {
            "ә": "а",
            "ғ": "г",
            "қ": "к",
            "ң": "н",
            "ө": "о",
            "ұ": "у",
            "ү": "у",
            "і": "и",
        }
        for k, v in replacements.items():
            name = name.replace(k, v)
        return name

    elif isinstance(name, list):
        return [normalize_kz(x) for x in name]

    return name
df["people_clean"] = df["people"].apply(normalize_kz)

In [172]:
df

,дата,время,рейтинг,текст,автор,likes_count,official_answer,отделение,люди,people,people_clean,people_final
1,2026-04-15,22:18:37,1,"Добрый день, приобрела билет в вашем приложени...",Наталья Ким,0,{'date_created': '2026-04-16T04:31:57.607004+0...,"Freedom Bank, Курмангазы",['Алматы-Сеул'],алматы-сеул,алматы-сеул,алматы-сеул
11,2026-04-08,11:33:48,1,Ужасное обслуживание. Не компетентные сотрудни...,Виктория Асаубаева,0,NaN,"Freedom Bank, Курмангазы","['Фридом', 'Турлов']",турлов,турлов,турлов
17,2026-04-06,19:37:04,5,Спасибо Айман за консультацию.,gulsim iglikova,0,{'date_created': '2026-04-07T05:38:45.216505+0...,"Freedom Bank, Курмангазы",['Айман'],айман,айман,айман
19,2026-04-04,16:55:10,1,"Посетили сегодня данный филиал. Во первых, неп...",Juliya Almazova,0,{'date_created': '2026-04-04T18:47:05.862607+0...,"Freedom Bank, Курмангазы",['Вай'],вай,вай,вай
20,2026-04-04,16:16:19,5,Кассадагы кызга рахмет .оте жаксы кызмет корсе...,Нурсултан Абрординов,0,{'date_created': '2026-04-04T18:32:54.456306+0...,"Freedom Bank, Курмангазы",['Дуйсенова'],дуйсенова айгерим,дуйсенова айгерим,дуйсенова
...,...,...,...,...,...,...,...,...,...,...,...,...
5181,2025-02-11,21:07:00,5,Добрый вечер! Понравилось само отделение светл...,Алтынай Кулжанова,2,{'date_created': '2025-02-13T21:32:38.765107+0...,"Freedon Bank, Достык","['Жакупову Гульнару', 'Гульнара']",гульнара жакупова,гульнара жакупова,гульнара жакупова
5181,2025-02-11,21:07:00,5,Добрый вечер! Понравилось само отделение светл...,Алтынай Кулжанова,2,{'date_created': '2025-02-13T21:32:38.765107+0...,"Freedon Bank, Достык","['Жакупову Гульнару', 'Гульнара']",гульнара жакупова,гульнара жакупова,гульнара
5184,2025-02-11,12:40:36,5,Спасибо большое сотруднику банка Жакуповой Гул...,Жазира Сакадиева,2,{'date_created': '2025-02-13T21:31:03.604368+0...,"Freedon Bank, Достык",['Жакуповой Гульнара'],гульнаре жакупова,гульнаре жакупова,гульнаре жакупова
5185,2025-02-11,12:02:53,5,Мне нравится все 💚 Конаев 💝,Манара Саимова,1,{'date_created': '2025-02-13T21:28:44.572378+0...,"Freedon Bank, Достык",['Конаев'],конаев,конаев,конаев


In [169]:
df['people_clean'].value_counts().head(50)  

people_clean
жунусбаева асем          729
жайлаубаев дастан        178
алтаева жазира           159
дуйсенова айгерим         90
калибекова айнура         76
таншолпан ергазина        57
ерсултан амантай          54
турсагулова асем          51
омарова айгерим           43
кудайбергенова гаухар     38
ильяс                     36
байбосын акер             31
айзат                     31
еркебулан                 26
альбина махмуд            25
гульнара жакупова         24
айганым                   23
кожахметов айзат          22
акбота                    21
арайлым мажит             20
бахтияр                   19
айман                     17
берди ашып                17
мукажанова айгуль         15
алмас                     15
байболат айгерим          14
махмутжан                 13
айжан                     13
турлов                    13
адэль                     11
гордеева                  11
айдана                    10
ари тез                   10
нуркамал                  10
а

In [ ]:
manual_map = {
    "акботе":" акбота",
    "гаухара": "кудайбергенова гаухар",
    "гульнара": "гульнара жакупова",
    "жунусбаевой асем": "жунусбаева асем",
    "әсем": "жунусбаева асем",
    "асем": "жунусбаева асем",
    "дастан": "жайлаубаев дастан",
    "алтаевой жазире": "алтаева жазира",
    "ерсултан": "ерсултан амантай",
    "таншолпан ергазиной": "таншолпан ергазина",
    "таншолпан ергазмна" : "таншолпан ергазина",
    'ергазиной таншолпан': 'таншолпан ергазина',
    "айгерим": "дуйсенова айгерим",
    "дуйсенова": "дуйсенова айгерим",
    "айнуре": "калибекова айнура",
    "калибекова": "калибекова айнура",
    "айгерим омарова": "омарова айгерим",
    "омарова": "омарова айгерим",
    "стамшал ильяса": "ильяс",
    "арыстанбековна жазира": "алтаева жазира",
    "акер": "байбосын акер",
}
df["people_clean"] = df["people_clean"].map(manual_map).fillna(df["people_clean"])


In [ ]:
words_to_remove = {"банктин"}

def remove_words(name):
    name=name.strip()
    parts = name.split()
    parts = [p for p in parts if p not in words_to_remove]

    return " ".join(parts)
df["people_clean"] = df["people_clean"].apply(remove_words)

In [170]:
df['people']=df['people_clean']

In [150]:
import re

stop_words = {"хорошо", "оте", "рахмет", "фридом", "коп", "берди ашып", "өте", "рахмет", "фридом"}

def clean_name(name):
    if not isinstance(name, str):
        return None

    name = name.lower()

    # убрать переносы строк
    name = name.replace("\n", " ")

    # убрать мусор слова
    words = name.split()
    words = [w for w in words if w not in stop_words]
    name = " ".join(words)

    return name.strip()
def normalize_order(name):
    if not isinstance(name, str):
        return name

    parts = name.split()

    if len(parts) == 2:
        # приводим к "фамилия имя"
        if len(parts[0]) <= len(parts[1]):
            return f"{parts[1]} {parts[0]}"

    return name
def full_clean(name):
    name = clean_name(name)
    name = normalize_order(name)
    
    return name
df["people_clean"] = df["people_clean"].apply(full_clean)
df = df.dropna(subset=["people_clean"])


In [96]:
from rapidfuzz import process, fuzz

names = df["people"].unique()
mapping = {}
canonical = []

for name in names:
    if not canonical:
        canonical.append(name)
        mapping[name] = name
        continue

    match, score, _ = process.extractOne(name, canonical, scorer=fuzz.WRatio)

    if score > 92:
        mapping[name] = match
    else:
        canonical.append(name)
        mapping[name] = name

df["people"] = df["people"].map(mapping)


In [166]:
df = df[df["people_clean"].notna()]
df = df[df["people_clean"].str.strip() != ""]

In [115]:
import re

def fix_cases(name):
    if not isinstance(name, str):
        return name

    # русские окончания фамилий
    name = re.sub(r"овой\b", "ова", name)
    name = re.sub(r"еву\b", "ев", name)
    name = re.sub(r"ову\b", "ов", name)
    name = re.sub(r"ину\b", "ин", name)

    # казахские окончания
    name = re.sub(r"(ға|ге|қа|ке|да|де|ты|ті|ны|ні)$", "", name)

    return name.strip()
def fix_order(name):
    if not isinstance(name, str):
        return name

    parts = name.split()

    if len(parts) == 2:
        # если второе слово выглядит как фамилия (часто длиннее)
        if len(parts[0]) < len(parts[1]):
            return f"{parts[1]} {parts[0]}"

    return name
def remove_patronymic(name):
    parts = name.split()
    if len(parts) == 3:
        return f"{parts[0]} {parts[1]}"
    return name
def final_clean(name):
    name = fix_cases(name)
    name = fix_order(name)
    name = remove_patronymic(name)
    return name
df["people_clean"] = df["people_clean"].apply(final_clean)


In [ ]:
from rapidfuzz import process, fuzz

names = df["people"].unique()
mapping = {}
canonical = []

for name in names:
    if not canonical:
        canonical.append(name)
        mapping[name] = name
        continue

    match, score, _ = process.extractOne(name, canonical, scorer=fuzz.WRatio)

    if score > 92:
        mapping[name] = match
    else:
        canonical.append(name)
        mapping[name] = name

df["people_clean"] = df["people_clean"].map(mapping)

In [84]:
def normalize_kz(name):
    if not isinstance(name, str):
        return name

    replacements = {
        "ә": "а",
        "ғ": "г",
        "қ": "к",
        "ң": "н",
        "ө": "о",
        "ұ": "у",
        "ү": "у",
        "і": "и",
    }

    for k, v in replacements.items():
        name = name.replace(k, v)

    return name
df["people_clean"] = df["people"].apply(normalize_kz)

In [86]:
df = df[df["people"].apply(lambda x: isinstance(x, list) and len(x) > 0)]

In [87]:
df["people"] = df["people"].apply(lambda x: list(set(x)))

In [76]:
df=df.explode("people")


In [174]:

import pandas as pd

cleaned = pd.read_csv('cleaned_reviews.csv', encoding='utf-8-sig')
result = pd.read_csv('result.csv', encoding='utf-8-sig')

# Джойним по дата+время+автор — эти три поля однозначно идентифицируют отзыв
key = ['дата', 'время', 'автор']
люди_map = cleaned[key + ['people_clean']].drop_duplicates(subset=key)

merged = result.merge(люди_map, on=key, how='left')

merged.to_csv('result1.csv', index=False, encoding='utf-8-sig')

In [2]:
import pandas as pd
df=pd.read_csv('result1.csv', encoding='utf-8-sig')

In [7]:

df


,дата,время,рейтинг,текст,автор,likes_count,official_answer,отделение,люди
0,2026-04-15,23:57:46,1,Долго обслуживают,Тимур Искаков,0,NaN,"Freedom Bank, Курмангазы",NaN
1,2026-04-15,22:18:37,1,"Добрый день, приобрела билет в вашем приложени...",Наталья Ким,0,{'date_created': '2026-04-16T04:31:57.607004+0...,"Freedom Bank, Курмангазы",алматы-сеул
2,2026-04-15,18:07:18,1,Очень медленно работают \nГрубят \nНет места д...,Мурат Хайдаров,0,{'date_created': '2026-04-16T04:29:03.021833+0...,"Freedom Bank, Курмангазы",NaN
3,2026-04-13,21:00:52,1,Просидел в очереди два часа при минимальном ко...,Grishkofan,0,NaN,"Freedom Bank, Курмангазы",NaN
4,2026-04-11,14:42:17,1,Добрый день! Сразу по сервису: \n- в комнатке ...,Alexandr Turdiyev,0,{'date_created': '2026-04-13T07:46:03.203688+0...,"Freedom Bank, Курмангазы",NaN
...,...,...,...,...,...,...,...,...,...
5193,2024-08-15,20:06:28,1,"ужасно обслуживание, менеджера элементарную по...",Берик Елеусизов,2,{'date_created': '2024-08-18T10:41:51.308397+0...,"Freedom Bank, Достык",NaN
5194,2024-08-06,12:16:33,2,Слабые менеджеры,Adlet Shakenov,2,{'date_created': '2024-08-06T21:35:40.561494+0...,"Freedom Bank, Достык",NaN
5195,2024-08-01,14:17:29,5,Хочу выразить благодарность кассиру Мукажанова...,Albina Marat,2,{'date_created': '2025-04-28T20:31:24.995252+0...,"Freedom Bank, Достык",мукажанова айгуль
5196,2024-07-24,17:00:41,5,Лучши Банк!,Apple User,1,{'date_created': '2024-07-24T20:55:03.041903+0...,"Freedom Bank, Достык",NaN


In [12]:
df.to_csv('result1.csv'
          , index=False, encoding='utf-8-sig')

In [181]:
df['отделение'].value_counts()

отделение
Freedom Bank, Курмангазы                                                 2040
Freedom Bank, Отделение по работе с юридическими и физическими лицами    1394
Freedom Bank, Шолохова                                                    942
Freedom Bank, Достык                                                      757
Freedom Bank, Абылай хана                                                  36
Freedom Bank, Отделение по обслуживанию VIP-клиентов                       29
Name: count, dtype: int64

In [180]:
dept_map = {
    "Freedon Bank, Шолохова": "Freedom Bank, Шолохова",
    "Freedon Bank, Достык": "Freedom Bank, Достык",
    "Freedon Bank, Абылай хана": "Freedom Bank, Абылай хана",

    
}
df["отделение"] = df["отделение"].replace(dept_map)

In [11]:
map = {
    "бакытханкызына асем": "жунусбаева асем",
    
}
df["люди"] = df["люди"].replace(map)

In [6]:
words_to_remove = {"банктин"}

def remove_words(name):
    if not isinstance(name, str):
        return name
    name=name.strip()
    parts = name.split()
    parts = [p for p in parts if p not in words_to_remove]

    return " ".join(parts)
df["люди"] = df["люди"].apply(remove_words)